In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())
print("Source directory:", PROJECT_ROOT / "src")

Working directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026
Source directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/src


In [2]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")

print("PG_CONN_STR configured:", bool(os.getenv("PG_CONN_STR")))

PG_CONN_STR configured: True


In [3]:
from index import text_search, vector_search, hybrid_search

print("Retrieval functions imported successfully.")

Retrieval functions imported successfully.


In [4]:
import psycopg

with psycopg.connect(os.environ["PG_CONN_STR"]) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                doc_id,
                symbol,
                year,
                quarter,
                COUNT(*) AS chunk_count
            FROM earnings_chunks
            GROUP BY doc_id, symbol, year, quarter
            ORDER BY symbol, year DESC, quarter DESC
        """)

        transcript_rows = cur.fetchall()

for row in transcript_rows:
    print(row)

('AAPL_2026Q3', 'AAPL', 2026, 3, 59)
('GOOG_2026Q2', 'GOOG', 2026, 2, 63)
('INTC_2026Q2', 'INTC', 2026, 2, 57)
('LRCX_2026Q4', 'LRCX', 2026, 4, 82)
('META_2026Q2', 'META', 2026, 2, 71)
('MSFT_2026Q4', 'MSFT', 2026, 4, 68)
('TSLA_2026Q2', 'TSLA', 2026, 2, 60)


In [5]:
# test retrieval manually
query = "What did Lam Research say about demand and outlook?"

for method_name, search_function in {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}.items():
    print(f"\n=== {method_name.upper()} ===")

    results = search_function(query, limit=5)

    for rank, result in enumerate(results, start=1):
        print(
            rank,
            result["doc_id"],
            result["symbol"],
            result["year"],
            result["quarter"],
            result.get("score", result.get("rrf_score")),
        )


=== TEXT ===
1 MSFT_2026Q4 MSFT 2026 4 0.0341959
2 MSFT_2026Q4 MSFT 2026 4 0.030396355
3 MSFT_2026Q4 MSFT 2026 4 0.030396355
4 MSFT_2026Q4 MSFT 2026 4 0.030396355
5 GOOG_2026Q2 GOOG 2026 2 0.030396355

=== VECTOR ===
1 LRCX_2026Q4 LRCX 2026 4 0.3583048070487501
2 INTC_2026Q2 INTC 2026 2 0.35780311478357885
3 LRCX_2026Q4 LRCX 2026 4 0.33826733599499903
4 LRCX_2026Q4 LRCX 2026 4 0.33057959907298096
5 MSFT_2026Q4 MSFT 2026 4 0.32082013363718254

=== HYBRID ===
1 MSFT_2026Q4 MSFT 2026 4 0.0341959
2 LRCX_2026Q4 LRCX 2026 4 0.3583048070487501
3 MSFT_2026Q4 MSFT 2026 4 0.030396355
4 INTC_2026Q2 INTC 2026 2 0.35780311478357885
5 MSFT_2026Q4 MSFT 2026 4 0.030396355


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.11.5 (main, Sep 11 2023, 08:19:27) [Clang 14.0.6 ]
/Users/camillecu/Downloads/KUL/llm_project/llm_project2026/llmproject311/bin/python


In [6]:
from evaluate_retrieval import (
    evaluate_search_function,
    evaluate_by_category,
)

# smoke test: evaluate each search function on a small sample of queries

In [ ]:

ground_truth = [
    {
        "query": "What did Lam Research say about demand?",
        "expected_doc_ids": {"LRCX_2026Q4"},
        "category": "company_name",
    },
    {
        "query": "What was LRCX management's outlook?",
        "expected_doc_ids": {"LRCX_2026Q4"},
        "category": "ticker",
    },
    {
        "query": "What risks did Lam Research discuss?",
        "expected_doc_ids": {"LRCX_2026Q4"},
        "category": "risk",
    },
]

In [11]:
from index import hybrid_search, text_search, vector_search
search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Hit Rate@5: {evaluation['hit_rate']:.3f}")
    print(f"MRR@5:      {evaluation['mrr']:.3f}")

    print("\nBy category:")
    print(evaluate_by_category(evaluation["details"]))


=== TEXT ===
Hit Rate@5: 0.000
MRR@5:      0.000

By category:
{'company_name': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}, 'ticker': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}, 'risk': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}}

=== VECTOR ===
Hit Rate@5: 0.333
MRR@5:      0.333

By category:
{'company_name': {'queries': 1, 'hit_rate': 1.0, 'mrr': 1.0}, 'ticker': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}, 'risk': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}}

=== HYBRID ===
Hit Rate@5: 0.333
MRR@5:      0.167

By category:
{'company_name': {'queries': 1, 'hit_rate': 1.0, 'mrr': 0.5}, 'ticker': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}, 'risk': {'queries': 1, 'hit_rate': 0.0, 'mrr': 0.0}}


In [12]:
for method_name, evaluation in evaluations.items():
    print(f"\n=== {method_name.upper()} failures ===")

    for row in evaluation["details"]:
        if row["hit"] == 0:
            print("Query:", row["query"])
            print("Expected:", row["expected_doc_ids"])
            print("Retrieved:", row["retrieved_doc_ids"])


=== TEXT failures ===
Query: What did Lam Research say about demand?
Expected: LRCX_2026Q4
Retrieved: MSFT_2026Q4|MSFT_2026Q4|GOOG_2026Q2|GOOG_2026Q2|GOOG_2026Q2
Query: What was LRCX management's outlook?
Expected: LRCX_2026Q4
Retrieved: META_2026Q2|GOOG_2026Q2|MSFT_2026Q4|META_2026Q2|AAPL_2026Q3
Query: What risks did Lam Research discuss?
Expected: LRCX_2026Q4
Retrieved: AAPL_2026Q3|MSFT_2026Q4|AAPL_2026Q3|TSLA_2026Q2|GOOG_2026Q2

=== VECTOR failures ===
Query: What was LRCX management's outlook?
Expected: LRCX_2026Q4
Retrieved: MSFT_2026Q4|MSFT_2026Q4|MSFT_2026Q4|MSFT_2026Q4|GOOG_2026Q2
Query: What risks did Lam Research discuss?
Expected: LRCX_2026Q4
Retrieved: INTC_2026Q2

=== HYBRID failures ===
Query: What was LRCX management's outlook?
Expected: LRCX_2026Q4
Retrieved: META_2026Q2|MSFT_2026Q4|GOOG_2026Q2|MSFT_2026Q4|MSFT_2026Q4
Query: What risks did Lam Research discuss?
Expected: LRCX_2026Q4
Retrieved: AAPL_2026Q3|INTC_2026Q2|MSFT_2026Q4|AAPL_2026Q3|TSLA_2026Q2


# full comparison

In [4]:
from evaluate_retrieval import load_ground_truth

ground_truth = load_ground_truth(
    "data/retrieval_ground_truth.csv"
)

# includes one intentionally ambiguous question: What did they say about margins?
# Its expected value is: AAPL_2026Q3|NEEDS_CONTEXT
# NEEDS_CONTEXT is not a transcript ID in the database. It means the system should ideally ask a follow-up question, such as: “Which company or earnings call do you mean?”

ground_truth_retrieval = [
    record
    for record in ground_truth
    if "NEEDS_CONTEXT" not in record["expected_doc_ids"]
]

print("All rows:", len(ground_truth))
print("Retrieval-evaluation rows:", len(ground_truth_retrieval))

All rows: 65
Retrieval-evaluation rows: 64


In [7]:
from index import text_search, vector_search, hybrid_search
from evaluate_retrieval import evaluate_search_function

search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth_retrieval,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Hit Rate@5: {evaluation['hit_rate']:.3f}")
    print(f"MRR@5:      {evaluation['mrr']:.3f}")



=== TEXT ===
Hit Rate@5: 0.969
MRR@5:      0.938

=== VECTOR ===
Hit Rate@5: 0.516
MRR@5:      0.467

=== HYBRID ===
Hit Rate@5: 0.969
MRR@5:      0.924
